# Step 3b — Prepare Clean Trades Data
Combine 3 days of trades per symbol and save for Person B

In [4]:
import pandas as pd
import glob
import re
import os

SYMBOLS  = ['WIFUSDT', 'ZAMAUSDT']
DATA_DIR = '../data'
OUT_DIR  = '../results'
os.makedirs(OUT_DIR, exist_ok=True)

# Auto-detect dates from filenames
files = glob.glob(f'{DATA_DIR}/{SYMBOLS[0]}_trades_*.parquet')
DATES = sorted(re.search(r'(\d{4}-\d{2}-\d{2})', f).group(1) for f in files)
print(f'Detected dates: {DATES}')

Detected dates: ['2026-04-12', '2026-04-13', '2026-04-14']


## 1. Combine, sort, and validate

In [5]:
def prepare_trades(symbol: str) -> pd.DataFrame:
    frames = []
    for date in DATES:
        df = pd.read_parquet(f'{DATA_DIR}/{symbol}_trades_{date}.parquet')
        frames.append(df)

    trades = pd.concat(frames).sort_index()

    # Convert index to tz-naive UTC (consistent with ground truth)
    trades.index = trades.index.tz_convert('UTC').tz_localize(None)

    # Drop redundant timestamp column
    trades = trades.drop(columns=['timestamp'])

    # Rename for clarity
    trades = trades.rename(columns={
        'price'  : 'price',
        'amount' : 'volume',
        'side'   : 'is_sell_aggressor'   # True = sell, False = buy
    })

    return trades

trades = {}
for symbol in SYMBOLS:
    trades[symbol] = prepare_trades(symbol)
    df = trades[symbol]
    print(f'\n=== {symbol} ===')
    print(f'Rows     : {len(df):,}')
    print(f'Period   : {df.index[0]}  →  {df.index[-1]}')
    print(f'Nulls    : {df.isnull().sum().sum()}')
    print(f'Neg price: {(df["price"] <= 0).sum()}')
    print(df.head(3))


=== WIFUSDT ===
Rows     : 55,607
Period   : 2026-04-11 22:00:12.243909317  →  2026-04-14 21:59:02.571355320
Nulls    : 0
Neg price: 0
                               price    volume  is_sell_aggressor
2026-04-11 22:00:12.243909317  0.201   1641.92              False
2026-04-11 22:00:12.243921501  0.201    695.17              False
2026-04-11 22:00:12.243923408  0.201  11868.66              False

=== ZAMAUSDT ===
Rows     : 641,585
Period   : 2026-04-11 22:00:02.405396069  →  2026-04-14 21:59:56.624069524
Nulls    : 0
Neg price: 0
                                price  volume  is_sell_aggressor
2026-04-11 22:00:02.405396069  0.0261   255.0              False
2026-04-11 22:00:03.348886737  0.0261   383.0              False
2026-04-11 22:00:03.497965954  0.0261  2345.0              False


## 2. Check time coverage vs ground truth

In [6]:
gt = {}
for symbol in SYMBOLS:
    gt[symbol] = pd.read_parquet(f'{OUT_DIR}/{symbol}_ground_truth_spread_1s.parquet')

for symbol in SYMBOLS:
    t_start = trades[symbol].index.min()
    t_end   = trades[symbol].index.max()
    g_start = gt[symbol].index.min()
    g_end   = gt[symbol].index.max()
    print(f'{symbol}')
    print(f'  trades   : {t_start}  →  {t_end}')
    print(f'  gt spread: {g_start}  →  {g_end}')

WIFUSDT
  trades   : 2026-04-11 22:00:12.243909317  →  2026-04-14 21:59:02.571355320
  gt spread: 2026-04-11 22:00:00  →  2026-04-14 21:59:59
ZAMAUSDT
  trades   : 2026-04-11 22:00:02.405396069  →  2026-04-14 21:59:56.624069524
  gt spread: 2026-04-11 22:00:00  →  2026-04-14 21:59:59


## 3. Save

In [7]:
for symbol in SYMBOLS:
    out_path = f'{OUT_DIR}/{symbol}_trades_clean.parquet'
    trades[symbol].to_parquet(out_path)
    print(f'Saved: {out_path}')

Saved: ../results/WIFUSDT_trades_clean.parquet
Saved: ../results/ZAMAUSDT_trades_clean.parquet
